In [5]:
import numpy as np
import pandas as pd
import polars as pl
from sklearn.metrics.pairwise import cosine_similarity

# Load data
df = pl.read_parquet("spy_10k_2015_present.parquet").to_pandas()

def semantic_distinctiveness_0_1(
    df: pd.DataFrame,
    embedding_col: str = "embedding",
    sector_col: str = "gics_sector",
    year_col: str = "filing_year",
    firm_col: str = "ticker",
    min_peers: int = 5
) -> pd.DataFrame:
    # Work on a copy
    work = df.copy()

    # Keep only rows with embeddings
    work = work.dropna(subset=[embedding_col]).reset_index(drop=True)

    # Force embeddings into numpy arrays
    work[embedding_col] = work[embedding_col].apply(
        lambda x: np.asarray(x, dtype=float)
    )

    out = []

    # Distinctiveness is computed within each sector-year peer group
    for (sector, year), grp in work.groupby([sector_col, year_col], dropna=False):
        grp = grp.reset_index(drop=True)

        # Not enough peers -> return NaN
        if len(grp) < min_peers:
            temp = grp[[firm_col, sector_col, year_col]].copy()
            temp["peer_count"] = len(grp) - 1
            temp["mean_cosine_distance"] = np.nan
            temp["distinctiveness_0_1"] = np.nan
            out.append(temp)
            continue

        # Stack embeddings
        X = np.vstack(grp[embedding_col].values)

        # Cosine similarity matrix
        sim = cosine_similarity(X)

        # Cosine distance matrix
        dist = 1.0 - sim

        mean_distances = []

        for i in range(len(grp)):
            # Exclude self-comparison
            peer_mask = np.ones(len(grp), dtype=bool)
            peer_mask[i] = False

            peer_dists = dist[i, peer_mask]
            mean_distances.append(peer_dists.mean())

        grp_out = grp[[firm_col, sector_col, year_col]].copy()
        grp_out["peer_count"] = len(grp) - 1
        grp_out["mean_cosine_distance"] = mean_distances

        # Min-max scaling to [0, 1] within this sector-year
        d_min = grp_out["mean_cosine_distance"].min()
        d_max = grp_out["mean_cosine_distance"].max()

        if np.isclose(d_min, d_max):
            # Everyone is effectively equally distinctive
            grp_out["distinctiveness_0_1"] = 0.0
        else:
            grp_out["distinctiveness_0_1"] = (
                (grp_out["mean_cosine_distance"] - d_min) / (d_max - d_min)
            )

        out.append(grp_out)

    return pd.concat(out, ignore_index=True)

distinct_df = semantic_distinctiveness_0_1(df)
distinct_df.head()

KeyError: ['embedding']

NameError: name 'semantic_distinctiveness_0_1' is not defined

ESGadj = ESG * (1 - KNN)